In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import lzma
import os

# ۱. تکرارپذیری (Reproducibility)
# تنظیم Seed باعث میشه در هر بار اجرا، وزن‌های اولیه مدل یکسان باشن تا بتونیم رفتار مدل رو تحلیل کنیم
torch.manual_seed(42)

# ۲. مدیریت سخت‌افزار (Device Configuration)
# بررسی وجود کارت گرافیک انویدیا برای انتقال محاسبات به GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ۳. هایپرپارامترهای مدل (Model Hyperparameters)
# این مقادیر متناسب با گرافیک 4060 تنظیم شده‌اند تا بهترین تعادل بین سرعت و دقت ایجاد شود
batch_size = 64          # تعداد جملاتی که به صورت موازی در یک مرحله پردازش می‌شوند
context_length = 256     # طول کانتکست (حداکثر طول توالی که مدل در هر لحظه می‌بیند)
max_iters = 5000         # تعداد کل مراحل آموزش (Training Steps)
learning_rate = 3e-4     # نرخ یادگیری ایده‌آل برای معماری‌های GPT (طبق استانداردهای OpenAI)
eval_interval = 500      # هر چند مرحله یک‌بار میزان خطای مدل ارزیابی شود
eval_iters = 200         # تعداد پارت‌هایی که برای میانگین‌گیری خطا استفاده می‌شوند

# ۴. ابعاد معماری شبکه ترانسفورمر
n_embd = 384             # بعد بردارهای امبدینگ (ویژگی‌های هر توکن)
n_head = 6               # تعداد هدهای مکانیزم Multi-Head Attention (هر هد ۶۴ بعد: ۳۸۴/۶)
n_layer = 6              # تعداد بلوک‌های متوالی ترانسفورمر (Transformer Blocks)
dropout = 0.2            # نرخ دراپ‌اوت برای جلوگیری از اورفیت شدن (Overfitting)

print(f"✅ Setup complete.")
print(f"🚀 Using device: {device}")
if device == 'cuda':
    print(f"🔥 GPU Model detected: {torch.cuda.get_device_name(0)}")

✅ Setup complete.
🚀 Using device: cuda
🔥 GPU Model detected: NVIDIA GeForce RTX 4060 Laptop GPU


In [5]:
# سلول ۲: خواندن بهینه بخشی از داده و ساخت توکنایزر سطح کاراکتر
import lzma

file_path = 'C:\\Users\\FH\\Documents\\PersionLLM\\PersianGPT\\data\\raw\\fa.txt.xz'

# تعیین سقف مجاز برای خواندن کاراکترها جهت جلوگیری از پر شدن ۱۸ گیگابایت رم سیستم
max_chars_to_read = 50_000_000 # ۵۰ میلیون کاراکتر برای آموزش یک مدل محلی کاملاً عالی و کافی است

print("⏳ در حال استریم کردن و خواندن بخشی از دیتاست بزرگ فارسی...")
text_buffer = []
chars_read = 0

# باز کردن فایل به صورت استریم (خط به خط) بدون خارج کردن کل فایل از حالت فشرده روی هارد
with lzma.open(file_path, mode='rt', encoding='utf-8') as f:
    for line in f:
        text_buffer.append(line)
        chars_read += len(line)
        if chars_read >= max_chars_to_read:
            break

# تبدیل خطوط خوانده شده به یک متن واحد و آزاد کردن بافر اولیه از حافظه رم
text = "".join(text_buffer)
del text_buffer 

print(f"✅ موفقیت‌آمیز: تعداد {len(text):,} کاراکتر فارسی با موفقیت در حافظه بارگذاری شد.")

# استخراج تمام کاراکترهای منحصربه‌فرد موجود در این بخش از متن (حروف، اعداد، علائم نگارشی)
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"🔤 تعداد اعضای واژه‌نامه (Vocab Size): {vocab_size}")

# ساخت مپینگ (نگاشت) دوطرفه: کاراکتر به عدد و عدد به کاراکتر
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }

# تعریف توابع توکنایزر (با در نظر گرفتن امنیت خطای کاراکترهای ناشناخته)
encode = lambda s: [stoi[c] for c in s if c in stoi] 
decode = lambda l: ''.join([itos[i] for i in l])

# تست سلامت توکنایزر با یک جمله نمونه فارسی
test_phrase = "مهندسی هوش مصنوعی با پایتورچ"
encoded_sample = encode(test_phrase)
decoded_sample = decode(encoded_sample)

print("\n🧪 تست توکنایزر:")
print(f"متن اصلی: {test_phrase}")
print(f"کدگذاری شده (Tokens): {encoded_sample}")
print(f"رمزگشایی شده: {decoded_sample}")

⏳ در حال استریم کردن و خواندن بخشی از دیتاست بزرگ فارسی...
✅ موفقیت‌آمیز: تعداد 50,000,385 کاراکتر فارسی با موفقیت در حافظه بارگذاری شد.
🔤 تعداد اعضای واژه‌نامه (Vocab Size): 947

🧪 تست توکنایزر:
متن اصلی: مهندسی هوش مصنوعی با پایتورچ
کدگذاری شده (Tokens): [312, 314, 313, 295, 299, 366, 10, 314, 315, 300, 10, 312, 301, 313, 315, 305, 366, 10, 288, 287, 10, 348, 287, 366, 290, 315, 297, 349]
رمزگشایی شده: مهندسی هوش مصنوعی با پایتورچ


In [6]:
# سلول ۳: تبدیل داده‌ها به تنسور و ساخت تابع DataLoader

print("در حال تبدیل متن به تنسور پایتورچ (این مرحله ممکن است چند ثانیه طول بکشد)...")
# تبدیل کل متن به اعداد و قرار دادن در یک تنسور یک بعدی
data = torch.tensor(encode(text), dtype=torch.long)
print(f"شکل تنسور داده‌ها: {data.shape}, نوع داده: {data.dtype}")

# پاک کردن متغیر متنی اولیه از رم برای آزادسازی حافظه
del text 

# تقسیم داده‌ها: 90 درصد برای آموزش (Train)، 10 درصد برای ارزیابی و تست (Validation)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f"تعداد توکن‌های آموزش: {len(train_data):,}")
print(f"تعداد توکن‌های ارزیابی: {len(val_data):,}")

def get_batch(split):
    """
    تولید یک بچ تصادفی از داده‌ها برای آموزش یا ارزیابی.
    همیشه سعی می‌کنیم توابع را به ساده‌ترین و خواناترین شکل ممکن بنویسیم.
    """
    data_source = train_data if split == 'train' else val_data
    
    # تولید اندیس‌های تصادفی برای شروع توالی‌ها
    # ما به تعداد batch_size عدد تصادفی انتخاب می‌کنیم
    ix = torch.randint(len(data_source) - context_length, (batch_size,))
    
    # استخراج توالی‌های ورودی (X) و هدف (Y)
    # خروجی Y دقیقاً همان X است که یک کاراکتر به جلو شیفت پیدا کرده
    x = torch.stack([data_source[i : i + context_length] for i in ix])
    y = torch.stack([data_source[i + 1 : i + context_length + 1] for i in ix])
    
    # انتقال مستقیم داده‌ها به کارت گرافیک
    x, y = x.to(device), y.to(device)
    
    return x, y

# تست تابع تولید بچ
xb, yb = get_batch('train')
print("\n📦 تست تولید بچ داده:")
print(f"ابعاد ورودی (X): {xb.shape}")
print(f"ابعاد هدف (Y): {yb.shape}")
print(f"محل قرارگیری داده‌ها: {xb.device}")

در حال تبدیل متن به تنسور پایتورچ (این مرحله ممکن است چند ثانیه طول بکشد)...
شکل تنسور داده‌ها: torch.Size([50000385]), نوع داده: torch.int64
تعداد توکن‌های آموزش: 45,000,346
تعداد توکن‌های ارزیابی: 5,000,039

📦 تست تولید بچ داده:
ابعاد ورودی (X): torch.Size([64, 256])
ابعاد هدف (Y): torch.Size([64, 256])
محل قرارگیری داده‌ها: cuda:0


In [8]:
# سلول ۴: پیاده‌سازی مکانیزم Attention

class Head(nn.Module):
    """ یک هد تکی از مکانیزم Self-Attention """
    def __init__(self, head_size):
        super().__init__()
        # لایه‌های خطی برای تولید بردارهای Query، Key و Value
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        
        # ثبت یک ماتریس پایین‌مثلثی (ماسک) برای جلوگیری از دیدن کلمات آینده
        # استفاده از register_buffer باعث می‌شود این ماتریس جزو وزن‌های قابل آموزش نباشد
        self.register_buffer('tril', torch.tril(torch.ones(context_length, context_length)))
        
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)
        
        # محاسبه امتیازات توجه (Attention Scores)
        # ضرب نقطه‌ای q و k برای پیدا کردن میزان ارتباط کلمات
        wei = q @ k.transpose(-2, -1) * (C ** -0.5) # مقیاس‌دهی برای پایداری گرادیان‌ها
        
        # ماسک کردن کلمات آینده با بی‌نهایت منفی (تا در تابع سافت‌مکس صفر شوند)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        
        # اعمال وزن‌ها روی مقادیر (Values)
        v = self.value(x) # (B, T, head_size)
        out = wei @ v     # (B, T, head_size)
        return out

class MultiHeadAttention(nn.Module):
    """ اجرای چندین هد Attention به صورت موازی برای یادگیری الگوهای مختلف متن """
    def __init__(self, num_heads, head_size):
        super().__init__()
        # ساخت لیستی از هدها به تعداد num_heads
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        # لایه تصویرسازی برای ترکیب خروجی هدها و برگرداندن به ابعاد اصلی
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # خروجی تمام هدها را در بعد آخر (Channel) به هم می‌چسبانیم
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

# یک تست کوچک برای اطمینان از درست کار کردن لایه‌ها روی دیتای بچ شده ما
# --- بخش تست اصلاح شده ---
print("✅ کلاس‌های Attention با موفقیت تعریف شدند.")

# ۱. ساخت یک لایه امبدینگ موقت برای تبدیل اعداد خام به بردارهای ۳۸۴ بعدی
dummy_embedding = nn.Embedding(vocab_size, n_embd).to(device)

# ۲. عبور دادن داده خام از امبدینگ
embedded_xb = dummy_embedding(xb) 
print(f"ابعاد داده بعد از امبدینگ: {embedded_xb.shape}") # باید [64, 256, 384] باشد

# ۳. حالا داده‌های امبد شده را به لایه Attention می‌دهیم
head_size = n_embd // n_head
dummy_mha = MultiHeadAttention(n_head, head_size).to(device)
dummy_out = dummy_mha(embedded_xb) 

print(f"ابعاد خروجی MultiHeadAttention: {dummy_out.shape}")

✅ کلاس‌های Attention با موفقیت تعریف شدند.
ابعاد داده بعد از امبدینگ: torch.Size([64, 256, 384])
ابعاد خروجی MultiHeadAttention: torch.Size([64, 256, 384])
